# Orca Nano DPO — merge/export fix (v2)

DPO training itself already succeeded in the previous run (238 pairs, 60 steps,
adapter saved) — this notebook ONLY re-does the merge+GGUF-export step, which
failed with `NotImplementedError` inside Unsloth's `save_pretrained_merged()`.

Root cause: the previous notebook attached the DPO-trained adapter via plain
`PeftModel.from_pretrained(model, adapter_dir, is_trainable=True)` — that
bypasses Unsloth's own loading path and breaks its merge/GGUF helpers, which
expect a model created through `FastLanguageModel.from_pretrained()` /
`get_peft_model()` specifically.

Fix: load the saved adapter directly via `FastLanguageModel.from_pretrained(
model_name=<adapter_path>)` — Unsloth auto-detects the base model from the
adapter's own `adapter_config.json` and wraps everything with its patches
intact, so `save_pretrained_merged`/`save_pretrained_gguf` work normally.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

!pip install -q unsloth trl transformers datasets peft bitsandbytes accelerate

In [ ]:
import glob

adapter_matches = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
print('Adapter config:', adapter_matches)
if not adapter_matches:
    raise FileNotFoundError("adapter_dpo not found — attach the dataset with the DPO-trained adapter.")
adapter_dir = adapter_matches[0].rsplit('/', 1)[0]
print('Adapter dir:', adapter_dir)

## Load base model + adapter together, the Unsloth-native way

Passing the adapter directory itself as `model_name` — Unsloth reads its
`adapter_config.json` to find the base model and loads both correctly.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_dir,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print("[load] base + DPO adapter loaded via Unsloth's native path")

## Merge LoRA + export GGUF (done in /tmp, not /kaggle/working)

In [ ]:
import shutil

shutil.rmtree("/tmp/merged", ignore_errors=True)
shutil.rmtree("/tmp/gguf", ignore_errors=True)

print("[merge] merging LoRA adapters (in /tmp)...")
model.save_pretrained_merged("/tmp/merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to /tmp/merged")

print("[gguf] converting to GGUF q4_k_m (in /tmp)...")
model.save_pretrained_gguf("/tmp/gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved under /tmp")

In [ ]:
import glob, shutil, os

candidates = [f for f in glob.glob('/tmp/**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found in /tmp:', candidates)

if candidates:
    source_path = candidates[0]
    filename = os.path.basename(source_path)
    dest_path = f'/kaggle/working/{filename}'
    shutil.copy(source_path, dest_path)
    print(f"[export] copied to {dest_path}")
    print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")
else:
    print('No GGUF file found under /tmp — check the [gguf] cell above for errors.')